In [81]:
import re 
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem import LancasterStemmer,PorterStemmer
from wordcloud import wordcloud

In [73]:
df = pd.read_csv(r"D:\Innomatics\Git_Uploads\imdb_sentiment_analysis\data\IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6 MB


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [8]:
## Positive and Negative is equally distributed

### Text PreProcessing

In [9]:
sw = stopwords.words("english")
sw

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [11]:
## Separating the negative stopwords, as they carry meanung and sentiment

In [12]:
n_sw = ["not","couldn't", 'didn', "didn't", 'doesn', "doesn't", "aren't",'hadn', "hadn't", 'hasn', "hasn't", "don't",'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"]

In [13]:
p_sw = set(sw).difference(n_sw)

In [14]:
df.sample()

,review,sentiment
44882,"I live in Mexico City, so I have to suffer thr...",negative


In [15]:
## As we have the HTML tags in the review we have to remove them too

In [53]:
la_st = LancasterStemmer()
po_st = PorterStemmer()

In [74]:
## Text Preprosseing
def text_preprocessing(review):
    # Lower case converstion
    review = review.lower()
    # cleaning html tags
    review = re.sub(r"<.*>"," ",review)
    # Cleaing the puntuation 
    spec_char = r"[^a-z0-9!?\s]"
    review = re.sub(spec_char,"",review)
    # Tokenisation
    review = review.split()
    # Stopwords and Stemming-- Lancaster
    review = [la_st.stem(word) for word in review if word not in p_sw]
    review = " ".join(review)
    return review

In [76]:
df["review"] = df["review"].apply(text_preprocessing)

In [77]:
df["review"]

0        on review ment watch 1 oz episod youl hook rig...
1        wond littl produc real real com hom littl thin...
2        thought wond way spend tim hot sum weekend sit...
3        bas ther famy littl boy jak think ther zomby c...
4        pet mat lov tim money vis stun film watch mr m...
                               ...                        
49995    thought movy right good job wasnt cre origin f...
49996    bad plot bad dialog bad act idiot direct annoy...
49997    cathol taught paroch el schools nun taught jes...
49998    im going disagr prevy com sid maltin on second...
49999    on expect star trek movy high art fan expect m...
Name: review, Length: 50000, dtype: str

## Visualization